# AI Repair Agent

## Objective

This notebook analyzes quarantined records generated by the Silver layer using a Large Language Model (LLM).

Responsibilities:

- Read quarantined datasets
- Analyze unresolved anomalies
- Generate repair recommendations
- Assign confidence scores
- Create AI-approved dataset
- Merge approved records with Silver

## Step 1 - Import Libraries

In [0]:
import os
import json
import pandas as pd

from pyspark.sql.functions import *
from pyspark.sql.types import *

## Step 2 - Load Configuration

In [0]:
from utils.config import *

## Step 3 - Load Latest Silver Batch

In [0]:
silver_batches = sorted(os.listdir(SILVER_PATH))

LATEST_BATCH = silver_batches[-1]

print("Latest Silver Batch:")
print(LATEST_BATCH)

In [0]:
LATEST_SILVER_PATH = os.path.join(
    SILVER_PATH,
    LATEST_BATCH
)

print(LATEST_SILVER_PATH)

## Step 4 - Load Silver and Quarantine Datasets

In [0]:
bus_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "bus_gps")
)

emergency_silver_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "emergency")
)

bus_quarantine_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "bus_quarantine")
)

emergency_quarantine_df = spark.read.parquet(
    os.path.join(LATEST_SILVER_PATH, "emergency_quarantine")
)

In [0]:
print("="*60)
print("Bus Silver :", bus_silver_df.count())
print("Emergency Silver :", emergency_silver_df.count())
print("Bus Quarantine :", bus_quarantine_df.count())
print("Emergency Quarantine :", emergency_quarantine_df.count())

##Install AI SDKs

In [0]:
%pip install -q google-genai
%pip install -q groq


## Initialize AI Providers

In [0]:
from google import genai
from gemini_config import GEMINI_API_KEY

In [0]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini initialized successfully.")

In [0]:
from groq import Groq
from groq_config import GROQ_API_KEY

In [0]:
groq_client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq initialized successfully.")

## AI Provider Router

In [0]:
import json

def analyze_record(prompt):

    gemini_error = None

    # =====================================
    # Try Gemini First
    # =====================================
    try:

        response = client.models.generate_content(
            model="gemini-flash-latest",
            contents=prompt
        )

        content = response.text.strip()

        return json.loads(content)

    except Exception as e:

        gemini_error = str(e)
        print("⚠️ Gemini unavailable. Switching to Groq...")

    # =====================================
    # Try Groq
    # =====================================
    try:

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an AI Data Quality Engineer. "
                        "Reply ONLY with valid JSON. "
                        "Do not explain anything. "
                        "Do not think step by step. "
                        "Do not use markdown. "
                        "Do not use ```json. "
                        "Return exactly one JSON object."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        content = response.choices[0].message.content.strip()

        # Remove markdown if Groq accidentally adds it
        content = content.replace("```json", "")
        content = content.replace("```", "")
        content = content.strip()

        return json.loads(content)

    except Exception as groq_error:

        return {
            "recommendation": None,
            "confidence": 0,
            "reason": f"Gemini Error: {gemini_error} | Groq Error: {str(groq_error)}",
            "action": "ERROR"
        }

#Test AI Router

In [0]:
result = analyze_record(
    "Reply ONLY with this JSON: {\"recommendation\":\"READY\",\"confidence\":100,\"reason\":\"Test\",\"action\":\"AUTO_APPROVE\"}"
)

print(result)

## Historical Data

In [0]:
def get_reference_examples(record, n=5):

    history = (
        emergency_silver_df
        .filter(emergency_silver_df.zone != "Unknown")
        .filter(emergency_silver_df.incident_type == record["incident_type"])
        .filter(emergency_silver_df.severity == record["severity"])
        .limit(n)
        .toPandas()
    )

    # If there are not enough matching severity records,
    # fall back to incident type only.
    if len(history) < 3:

        history = (
            emergency_silver_df
            .filter(emergency_silver_df.zone != "Unknown")
            .filter(emergency_silver_df.incident_type == record["incident_type"])
            .limit(n)
            .toPandas()
        )

    examples = ""

    for _, row in history.iterrows():

        examples += f"""
Incident Type : {row['incident_type']}
Severity      : {row['severity']}
Response Time : {row['response_time']}
Status        : {row['status']}
Zone          : {row['zone']}

"""

    return examples

## AI Prompt Template

In [0]:
def build_prompt(record):

    examples = get_reference_examples(record)

    return f"""
You are an AI Data Quality Engineer.

Your job is to repair missing values using historical records.

====================================================
Historical Records
====================================================

{examples}

====================================================
Current Record
====================================================

Incident ID   : {record["incident_id"]}
Zone          : {record["zone"]}
Incident Type : {record["incident_type"]}
Severity      : {record["severity"]}
Response Time : {record["response_time"]}
Status        : {record["status"]}

====================================================
Task
====================================================

Compare the CURRENT RECORD with the HISTORICAL RECORDS.

Use the following attributes:

- Incident Type
- Severity
- Response Time
- Status

Predict the MOST LIKELY Zone.

If multiple Zones are equally likely,
choose the best one but LOWER the confidence.

====================================================
Output Format
====================================================

Return ONLY valid JSON.

{{
    "recommendation": "<Zone Name>",
    "confidence": <integer>,
    "reason": "<one short sentence>",
    "action": "<AUTO_APPROVE or HUMAN_REVIEW>"
}}

====================================================
Confidence Guidelines
====================================================

95-100 : Almost identical historical records.

85-94 : Strong evidence.

70-84 : Reasonable evidence.

50-69 : Weak evidence.

====================================================
Rules
====================================================

1. Confidence must be an integer between 0 and 100.

2. If confidence >= 80
   action = "AUTO_APPROVE"

3. Otherwise
   action = "HUMAN_REVIEW"

4. Reason must be ONE short sentence.

5. Do NOT explain your reasoning.

6. Do NOT use markdown.

7. Do NOT use ```json.

8. Return ONLY the JSON object.

"""

## AI Batch Processing

In [0]:
from datetime import datetime
import json

ai_audit_records = []

quarantine_records = (
    emergency_quarantine_df
    .limit(10)
    .toPandas()
    .to_dict("records")
)

print("Total Quarantine Records :", len(quarantine_records))
#emergency_quarantine_df.toPandas().to_dict("records") this is for all records but due to gemini limitation testing on 10

print("Total Quarantine Records :", len(quarantine_records))

In [0]:
for record in quarantine_records:

    try:

        # Build Prompt
        prompt = build_prompt(record)

        # Analyze using AI Router (Gemini -> Groq Fallback)
        result = analyze_record(prompt)
        print("=" * 80)
        print(record["incident_id"])
        print(result)

        # Store AI Audit
        ai_audit_records.append({

            "incident_id": record["incident_id"],

            "recommendation": result["recommendation"],

            "confidence": result["confidence"],

            "action": (
                "AUTO_APPROVE"
            if result["confidence"] >= 80
            else "HUMAN_REVIEW"

            ),

            "reason": result["reason"],

            "processed_at": datetime.now(),

            "model": "Gemini/Groq Router"

        })

    except Exception as e:

        ai_audit_records.append({

            "incident_id": record["incident_id"],

            "recommendation": None,

            "confidence": 0,

            "action": "ERROR",

            "reason": str(e),

            "processed_at": datetime.now(),

            "model": "Gemini/Groq Router"

        })

In [0]:
for i in range(3):
    print("=" * 80)
    print(build_prompt(quarantine_records[i]))

## Create AI Audit DataFrame

In [0]:
print("AI Records Processed :", len(ai_audit_records))

In [0]:
ai_audit_df = pd.DataFrame(ai_audit_records)
ai_audit_df.head()

In [0]:
ai_audit_records[0]

##AI Decision Summary

In [0]:
print("=" * 60)
print("AI Decision Summary")
print("=" * 60)

print("AI Approved :", len(ai_audit_df[ai_audit_df["action"] == "AUTO_APPROVE"]))
print("Human Review :", len(ai_audit_df[ai_audit_df["action"] == "HUMAN_REVIEW"]))

## AI Approved Dataset

In [0]:
ai_approved_df = ai_audit_df[
    ai_audit_df["action"] == "AUTO_APPROVE"
]
print("AI Approved Records :", len(ai_approved_df))
ai_approved_df

## Prepareing Records for Merge

In [0]:
approved_records = []

for record in quarantine_records:

    audit = ai_approved_df[
        ai_approved_df["incident_id"] == record["incident_id"]
    ]
    if len(audit) == 0:
        continue
    audit = audit.iloc[0]
    # Create a COPY so we don't modify the original record
    repaired_record = record.copy()
    repaired_record["zone"] = audit["recommendation"]
    repaired_record["repair_status"] = "AI_REPAIRED"
    repaired_record["repair_reason"] = audit["reason"]
    repaired_record["repair_confidence"] = int(audit["confidence"])

    approved_records.append(repaired_record)

print("Records Ready For Merge :", len(approved_records))

## Create Spark DataFrame

In [0]:
import pandas as pd

approved_pdf = pd.DataFrame(approved_records)

# Convert timestamp columns
approved_pdf["timestamp"] = pd.to_datetime(approved_pdf["timestamp"])
approved_pdf["ingestion_timestamp"] = pd.to_datetime(
    approved_pdf["ingestion_timestamp"]
)

approved_spark_df = spark.createDataFrame(approved_pdf)

approved_spark_df.printSchema()

# Align the schema

In [0]:
silver_columns = emergency_silver_df.columns

approved_spark_df = (
    approved_spark_df
    .select(*silver_columns)
)

## Merge AI Approved Records with Silver Layer

In [0]:
approved_ids = [
    row["incident_id"]
    for row in approved_records
]

updated_emergency_silver_df = (
    emergency_silver_df
    .filter(~emergency_silver_df.incident_id.isin(approved_ids))
    .unionByName(approved_spark_df)
)

print("Updated Silver Records :", updated_emergency_silver_df.count())

# Merge with Silver

In [0]:
approved_ids = [
    row["incident_id"]
    for row in approved_records
]
updated_emergency_silver_df = (
    emergency_silver_df
    .filter(~col("incident_id").isin(approved_ids))
    .unionByName(approved_spark_df)
)
print("Updated Silver Records :", updated_emergency_silver_df.count())

## Save Updated Silver Layer

In [0]:
updated_emergency_silver_df.write.mode("overwrite").parquet(
    os.path.join(
        LATEST_SILVER_PATH,
        "emergency"
    )
)

print("✅ Updated Silver Layer Saved Successfully")

## Pipeline Summary

In [0]:
print("=" * 70)
print("SMART CITY AI SELF-HEALING PIPELINE")
print("=" * 70)

print(f"Emergency Silver Records : {emergency_silver_df.count()}")
print(f"Quarantine Records       : {len(quarantine_records)}")
print(f"AI Approved              : {len(ai_approved_df)}")
print(f"Human Review             : {len(quarantine_records) - len(ai_approved_df)}")
print(f"Merged Back to Silver    : {len(ai_approved_df)}")

print("=" * 70)
print("PIPELINE EXECUTED SUCCESSFULLY")
print("=" * 70)

In [0]:
print(emergency_silver_df.columns)
print(approved_spark_df.columns)

In [0]:
emergency_silver_df.printSchema()

approved_spark_df.printSchema()

##Display AI Repaired Records

In [0]:
ai_approved_df[
    ["incident_id", "recommendation", "confidence", "action"]
]

In [0]:
updated_emergency_silver_df.filter(
    col("repair_status") == "AI_REPAIRED"
).select(
    "incident_id",
    "zone",
    "repair_status",
    "repair_confidence"
).show(truncate=False)